<a href="https://colab.research.google.com/github/YevheniiaMalchenko/ML/blob/main/%D0%9B%D0%A0_9_%D0%9C%D0%B0%D0%BB%D1%8C%D1%87%D0%B5%D0%BD%D0%BA%D0%BE_3_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** _Мальченко Євгенія Олександрівна_________________________  
**Група:** ________3-10_________________________  
**Дата:** ____10.11.2026______________________________

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [1]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [2]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [3]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:

In [4]:
# - кількість унікальних користувачів;
ratings_df['user_id'].nunique()

943

In [5]:
# - кількість унікальних фільмів;
ratings_df['item_id'].nunique()

1682

In [7]:
# - мінімальне, максимальне, середнє значення rating;
print(ratings_df['rating'].min())
print(ratings_df['rating'].max())
print(ratings_df['rating'].mean())

1.0
5.0
3.52986


In [8]:
# - пропущені значення (за наявності).
ratings_df.isnull().sum()

,0
user_id,0
item_id,0
rating,0
timestamp,0


**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

Набір даних містить оцінки від 943 унікальних користувачів для 1682 різних фільмів. Оцінки виставляються за шкалою від 1 до 5, а середнє значення становить приблизно 3.53, що свідчить про те, що користувачі загалом схильні ставити оцінки вище середнього. Пропущених значень у даних немає.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [9]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42
)

In [13]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
# 1. reader = Reader(rating_scale=...)
reader = Reader(rating_scale=data.reader.rating_scale)
# 2. train_data = Dataset.load_from_df(...)
train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
testset = list(test_df[['user_id', 'item_id', 'rating']].itertuples(index=False, name=None))

# 4. Перевірте розміри train/test.
print(f"Розмір train: {train_df.shape}")
print(f"Розмір test: {test_df.shape}")

Розмір train: (80000, 4)
Розмір test: (20000, 4)


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [14]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

Для SVD:
n_factors (50, 100): кількість прихованих ознак. Більше число дозволяє моделі знайти складніші зв'язки, але працює повільніше.
lr_all (0.005, 0.01): швидкість навчання. Малі значення забезпечують стабільне навчання без різких стрибків.
reg_all (0.02, 0.05): захист від перенавчання. Не дає моделі зазубрити дані, щоб вона краще працювала на нових.

Для KNNBasic:
k (20, 40). Більше число робить прогноз більш згладженим та стійким.
min_k (1, 5): мінімальна кількість сусідів, потрібна для того, щоб взагалі зробити прогноз.
sim_options: обираємо, як саме шукати схожість (метрика cosine чи pearson) та за яким принципом (шукати схожих людей user_based=True чи схожі фільми user_based=False). Це дозволяє порівняти різні підходи.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [15]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_svd.fit(train_data)

In [16]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1
)
grid_search_knn.fit(train_data)

In [18]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print("Результати для SVD")
print(f"Найкращий RMSE: {grid_search_svd.best_score['rmse']:.4f}")
print(f"Найкращий MAE: {grid_search_svd.best_score['mae']:.4f}")
print(f"Найкращі параметри: {grid_search_svd.best_params['rmse']}")

print("\nРезультати для KNNBasic")
print(f"Найкращий RMSE: {grid_search_knn.best_score['rmse']:.4f}")
print(f"Найкращий MAE: {grid_search_knn.best_score['mae']:.4f}")
print(f"Найкращі параметри: {grid_search_knn.best_params['rmse']}")

Результати для SVD
Найкращий RMSE: 0.9428
Найкращий MAE: 0.7438
Найкращі параметри: {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}

Результати для KNNBasic
Найкращий RMSE: 1.0273
Найкращий MAE: 0.8145
Найкращі параметри: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9428 | 0.7438| {'n_factors': 100, 'lr_all': 0.01, 'reg_all': 0.05}|
| KNNBasic |1.0273 | 0.8145 | {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}} |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

За результатами крос-валідації, модель SVD показала кращу якість порівняно з KNNBasic. Найкращий показник RMSE для SVD становить 0.9428, тоді як для KNNBasic ці показники вищі: RMSE - 1.0273. Це свідчить про те, що матрична факторизація краще вловлює приховані патерни в уподобаннях користувачів, ніж пошук найближчих сусідів.

Для моделі SVD найкращими виявилися параметри: 100 прихованих факторів, швидкість навчання lr_all=0.01 та регуляризація reg_all=0.05.
Для моделі KNNBasic найкращими стали: 40 сусідів (k=40), мінімум 1 сусід (min_k=1), косинусна міра схожості (name='cosine') та пошук схожих користувачів (user_based=True)

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [20]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']
# TODO 6.2 — Створіть trainset із train_data.
trainset = train_data.build_full_trainset()
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.
best_svd = SVD(
    n_factors=best_svd_params['n_factors'],
    lr_all=best_svd_params['lr_all'],
    reg_all=best_svd_params['reg_all'],
    random_state=42
)

best_knn = KNNBasic(
    k=best_knn_params['k'],
    min_k=best_knn_params['min_k'],
    sim_options=best_knn_params['sim_options']
)
best_svd.fit(trainset)
best_knn.fit(trainset)

print("успішно навчені")

Computing the cosine similarity matrix...
Done computing similarity matrix.
успішно навчені


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [21]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)



In [22]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

results = results.sort_values(by='RMSE').reset_index(drop=True)
display(results.round(4))

,Model,RMSE,MAE
0,SVD,0.9194,0.7247
1,KNNBasic,1.0120,0.7984


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.9194 |0.7247|
| KNNBasic | 1.0120 | 0.7984 |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

Оцінки суттєво не відрізняються. Оцінки на незалежній тестовій вибірці виявилися дуже близькими до результатів крос-валідації. Для моделі SVD тестовий RMSE становить 0.9194 (проти 0.9428 на CV), а для KNNBasic - 1.0120 (проти 1.0273 на CV). Навіть спостерігається невелике покращення (менші значення помилок) на тесті. Це свідчить про те, що моделі не перенавчилися під час крос-валідації. Модель SVD стабільно показує кращий результат, ніж KNNBasic, як на крос-валідації, так і на тесті.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [27]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
if grid_search_svd.best_score['rmse'] < grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'

print(f"Обрана модель для рекомендацій: {best_model_name}")

# TODO 8.2 — Перевірте, що користувач присутній у trainset.
try:
    inner_uid = trainset.to_inner_uid(user_id)
    print(f"Користувач {user_id} успішно знайдений у trainset.")

    # TODO 8.3 — Знайдіть неоцінені ним фільми.
    user_rated_items = set()
    for iid, _ in trainset.ur[inner_uid]:
        user_rated_items.add(trainset.to_raw_iid(iid))

    all_items = set(ratings_df['item_id'].astype(str).unique())

    unrated_items = list(all_items - user_rated_items)

    # TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
    predictions = []
    for item_id in unrated_items:
        pred = best_model.predict(uid=user_id, iid=item_id)
        predictions.append((item_id, pred.est))

    recommendations_df = pd.DataFrame(predictions, columns=['item_id', 'predicted_rating'])
    top_10 = recommendations_df.sort_values(by='predicted_rating', ascending=False).head(10)

    print(f"\nТоп-10 рекомендацій для користувача {user_id}:")
    display(top_10)

except ValueError:
    print(f"Помилка: користувача з ID '{user_id}' немає у навч. вибірці.")

Обрана модель для рекомендацій: SVD
Користувач 196 успішно знайдений у trainset.

Топ-10 рекомендацій для користувача 196:


,item_id,predicted_rating
34,318,4.578034
721,427,4.510453
936,603,4.455609
250,1169,4.441489
1467,661,4.414566
951,479,4.390852
46,513,4.356216
918,357,4.324538
242,64,4.322364
1157,313,4.300572


## 9. Перевірка виконання

Позначте після реалізації:

- [+] Дані `ml-100k` завантажено та перевірено.
- [+] Є розбиття 80/20 з фіксованим `random_state`.
- [+] `Reader`, `train_data` і `testset` створені коректно.
- [+] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [+] Для SVD і KNNBasic показано параметри та результати CV.
- [+] Моделі навчено на повній навчальній частині.
- [+] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [+] Наведено порівняльну таблицю та інтерпретацію.
- [+] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [+] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

**Висновок**

За результатами крос-валідації для подальшої роботи було обрано модель SVD, оскільки вона показала меншу помилку (RMSE = 0.94), ніж KNNBasic (RMSE = 1.03). Це означає, що SVD краще знаходить приховані зв'язки між смаками користувачів та фільмами.

Найкращі гіперпараметри для SVD виявилися такими:
100 прихованих факторів,
швидкість навчання 0.01
регуляризація 0.05.
Для KNNBasic найкращими стали:
40 сусідів,
мінімум 1 сусід та косинусна міра схожості при пошуку схожих користувачів.

Підсумкові результати на незалежній тестовій вибірці для SVD становлять RMSE = 0.9194 та MAE = 0.7247, а для KNNBasic - RMSE = 1.0120 та MAE = 0.7984. Різниці між результатами крос-валідації та тестування практично немає (помилки навіть трохи зменшилися), що свідчить про відсутність перенавчання і хорошу здатність моделей узагальнювати нові дані.

Головне обмеження отриманих рекомендацій полягає в тому, що алгоритми не можуть будувати якісні прогнози для абсолютно нових користувачів або нових фільмів, яких не було в навчальних даних. Система просто не матиме історії оцінок, щоб знайти схожих людей або закономірності.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**